In [16]:
import requests
import pandas as pd
import time

In [17]:
url = f"{"https://eodhistoricaldata.com/api"}/exchange-symbol-list/JK?api_token={"697f422d103f18.06682771"}&fmt=json"

response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"Failed to fetch Indonesia tickers: {response.status_code}")

tickers_data = response.json()

df_tickers = pd.DataFrame(tickers_data)

print("Total Indonesia Stocks:", len(df_tickers))
df_tickers.head()

Exception: Failed to fetch Indonesia tickers: 402

In [ ]:
symbols = df_tickers['Code'].dropna().tolist()

print("Total symbols:", len(symbols))
print("Sample symbols:", symbols[:20])

Total symbols: 961
Sample symbols: ['AADI', 'AALI', 'ABBA', 'ABDA', 'ABMM', 'ACES', 'ACRO', 'ACST', 'ADCP', 'ADES', 'ADHI', 'ADMF', 'ADMG', 'ADMR', 'ADRO', 'AEGS', 'AGAR', 'AGII', 'AGRO', 'AGRS']


In [ ]:
all_data = []

for i, symbol in enumerate(symbols):
    try:
        print(f"Fetching {symbol} ({i+1}/{len(symbols)})")

        url = f"{"https://eodhistoricaldata.com/api"}/eod/{symbol}.JK?api_token={"697f422d103f18.06682771"}&fmt=json&from=2018-01-01&to=2024-12-31"

        response = requests.get(url)
        
        if response.status_code != 200:
            print(f"Failed: {symbol}")
            continue

        data = response.json()

        if isinstance(data, list) and len(data) > 0:
            df = pd.DataFrame(data)
            df['Stock'] = symbol
            all_data.append(df)

        time.sleep(0.3)

    except Exception as e:
        print(f"Error with {symbol}: {e}")

Fetching AADI (1/961)
Fetching AALI (2/961)
Fetching ABBA (3/961)
Fetching ABDA (4/961)
Fetching ABMM (5/961)
Fetching ACES (6/961)
Fetching ACRO (7/961)
Fetching ACST (8/961)
Fetching ADCP (9/961)
Fetching ADES (10/961)
Fetching ADHI (11/961)
Fetching ADMF (12/961)
Fetching ADMG (13/961)
Fetching ADMR (14/961)
Fetching ADRO (15/961)
Fetching AEGS (16/961)
Fetching AGAR (17/961)
Fetching AGII (18/961)
Fetching AGRO (19/961)
Fetching AGRS (20/961)
Fetching AHAP (21/961)
Fetching AIMS (22/961)
Fetching AISA (23/961)
Fetching AKKU (24/961)
Fetching AKPI (25/961)
Fetching AKRA (26/961)
Fetching AKSI (27/961)
Fetching ALDO (28/961)
Fetching ALII (29/961)
Fetching ALKA (30/961)
Fetching ALMI (31/961)
Fetching ALTO (32/961)
Fetching AMAG (33/961)
Fetching AMAN (34/961)
Fetching AMAR (35/961)
Fetching AMFG (36/961)
Fetching AMIN (37/961)
Fetching AMMN (38/961)
Fetching AMMS (39/961)
Fetching AMOR (40/961)
Fetching AMRT (41/961)
Fetching ANDI (42/961)
Fetching ANJT (43/961)
Fetching ANTM (44/96

In [ ]:
if len(all_data) == 0:
    raise Exception("No data collected!")

final_df = pd.concat(all_data, ignore_index=True)

print("Total rows:", len(final_df))
final_df.head()

Total rows: 518


,date,open,high,low,close,adjusted_close,volume,warning,Stock
0,2025-04-09,5750.0,5925.0,5700.0,5750.0,5396.4046,12763900.0,Data is limited by one year as you have free s...,AADI
1,2025-04-09,5225.0,5375.0,5200.0,5275.0,5034.7838,683000.0,Data is limited by one year as you have free s...,AALI
2,2025-04-09,16.0,16.0,16.0,16.0,16.0000,285300.0,Data is limited by one year as you have free s...,ABBA
3,2025-04-09,3400.0,3400.0,3400.0,3400.0,3400.0000,0.0,Data is limited by one year as you have free s...,ABDA
4,2025-04-09,2850.0,2900.0,2800.0,2830.0,2698.7909,2323900.0,Data is limited by one year as you have free s...,ABMM


In [ ]:
final_df['date'] = pd.to_datetime(final_df['date'])

# Remove duplicates
final_df = final_df.drop_duplicates()

# Remove missing values
final_df = final_df.dropna()

print("Cleaned data shape:", final_df.shape)

Cleaned data shape: (510, 9)


In [ ]:
# Drop only if too many missing values (e.g., >50%)
threshold = int(0.5 * len(pivot_close))

pivot_close = pivot_close.dropna(axis=1, thresh=threshold)

# Fill remaining missing values
pivot_close = pivot_close.fillna(method='ffill').fillna(method='bfill')

print("Final stock count:", len(pivot_close.columns))

Final stock count: 510


C:\Users\ivanf\AppData\Local\Temp\ipykernel_11660\2764144787.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  pivot_close = pivot_close.fillna(method='ffill').fillna(method='bfill')


In [ ]:
pivot_close.to_csv("idx_stock_final_clean.csv")
df_tickers.to_csv("idx_all_tickers.csv", index=False)

print("Data saved successfully!")

Data saved successfully!


In [ ]:
print("===== SUMMARY =====")
print("Total IDX Stocks (raw):", len(symbols))
print("Total Valid Stocks (clean):", len(pivot_close.columns))
print("Total Time Points:", pivot_close.shape[0])

===== SUMMARY =====
Total IDX Stocks (raw): 961
Total Valid Stocks (clean): 510
Total Time Points: 1


In [ ]:
final_df.to_csv("idx_stock_raw_full.csv", index=False)
print("Raw dataset saved!")

Raw dataset saved!
